# Modern nanoGPT — dual-T4 training

Trains the RoPE / RMSNorm ablations on Kaggle's 2x T4 accelerator, **one model per GPU**,
so two architectures train simultaneously rather than one after the other.

This is not DDP. DDP would make both GPUs cooperate on a single model; here each GPU owns
an independent run with its own `CUDA_VISIBLE_DEVICES`, which is the right call when the
goal is comparing architectures rather than training one model faster.

**Before running:** Settings -> Accelerator -> **GPU T4 x2**, and Internet -> **On**.

Session limits worth planning around: ~12h per session, 30h/week quota. Every run
checkpoints and is launched with `--resume`, so a session timeout costs you nothing —
re-run this notebook and each run picks up where it stopped.

## 1. Environment

In [ ]:
import torch, os, sys, subprocess, time, shutil, glob, json, math

print("torch", torch.__version__)
n_gpu = torch.cuda.device_count()
print("GPUs visible:", n_gpu)
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"  [{i}] {p.name}  {p.total_memory/1e9:.1f} GB  SM {p.major}.{p.minor}")
# CAREFUL: torch.cuda.is_bf16_supported() defaults to including_emulation=True and
# returns True on Turing (T4, SM 7.5), which has fp16 tensor cores but NO bf16 ones.
# Trusting it selects a path with no tensor cores at all -- measured ~10x slower here.
# The trainer picks by compute capability instead; bf16 hardware starts at SM 8.0.
if n_gpu:
    _major = torch.cuda.get_device_capability()[0]
    print("is_bf16_supported() says:", torch.cuda.is_bf16_supported(), "  <- do not trust")
    print("real bf16 tensor cores  :", _major >= 8, f"(SM {_major}.x)")
    print("=> trainer will use     :", "bfloat16" if _major >= 8 else "float16 + GradScaler")
if n_gpu < 2:
    print("\nWARNING: fewer than 2 GPUs. Set Accelerator to 'GPU T4 x2' to train two at once.")

In [ ]:
!pip install -q tiktoken datasets

## 2. Get the code

Three ways, tried in order: files already alongside the notebook, a git clone, or a Kaggle
Dataset you attached. Set `REPO_URL` if you want the clone path.

In [ ]:
REPO_URL = ""   # e.g. "https://github.com/<you>/modern-nanogpt.git"
WORK = "/kaggle/working/modern-nanogpt"
NEEDED = ["train_modern_gpt.py", "train_gpt2.py", "benchmark.py", "fineweb.py", "hellaswag.py"]

def have_all(d):
    return all(os.path.exists(os.path.join(d, f)) for f in NEEDED)

if have_all(os.getcwd()):
    WORK = os.getcwd()
elif REPO_URL:
    if not os.path.exists(WORK):
        subprocess.run(["git", "clone", "--depth=1", REPO_URL, WORK], check=True)
else:
    # look for the files in any attached dataset and copy them somewhere writable
    hits = glob.glob("/kaggle/input/**/train_modern_gpt.py", recursive=True)
    assert hits, "Set REPO_URL, or attach the repo as a Kaggle Dataset via 'Add Input'."
    src = os.path.dirname(hits[0])
    os.makedirs(WORK, exist_ok=True)
    for f in NEEDED + ["input.txt"]:
        if os.path.exists(os.path.join(src, f)):
            shutil.copy(os.path.join(src, f), WORK)

os.chdir(WORK)
assert have_all(WORK), f"missing files in {WORK}: {[f for f in NEEDED if not os.path.exists(f)]}"
print("working directory:", WORK)
print(sorted(os.listdir(WORK))[:20])

## 3. Correctness checks

Run these first — they take seconds and they are what makes the comparison trustworthy.
The parity check proves `--pos=learned --norm=layernorm` is bit-identical to the original
`train_gpt2.py`, so any loss gap later is the architecture and not the random seed.

In [ ]:
!python benchmark.py --mode=check

## 4. Smoke test

30 seconds on `input.txt`, no download. Validates the whole loop — data, autocast dtype,
GradScaler, checkpointing — before spending GPU quota.

In [ ]:
!python train_modern_gpt.py --data_dir=input.txt --run_name=smoke \
    --pos=rope --norm=rmsnorm --n_layer=2 --n_head=4 --n_embd=128 --block_size=128 \
    --batch_size=4 --seq_len=128 --total_batch_size=2048 --max_steps=20 \
    --warmup_steps=5 --eval_every=10 --val_steps=2

## 5. Data

**Expected setup: attach your uploaded Dataset via `Add Input`.** Build the shards once on
your own machine and upload them:

```bash
python fineweb.py --shards 7 --local_dir C:/data/edu_fineweb10B   # not in a synced folder
kaggle datasets init -p C:/data/edu_fineweb10B                    # edit title + id
kaggle datasets create -p C:/data/edu_fineweb10B
```

The cell below finds anything matching `/kaggle/input/**/edufineweb_*.npy` on its own, so
no path editing is needed. `/kaggle/working` is wiped when a session ends and these runs
span several sessions, so downloading here instead would re-pay the download *and* the
tokenization every time — the in-notebook download is a last-resort fallback.

Sizing: shard 0 is validation, the rest are training, 100M tokens / 200MB each. At
`MAX_STEPS=4000 x 131072` a run consumes ~524M tokens, so 6 training shards keeps it to a
single epoch. Scale shards with steps.

In [ ]:
SHARDS = 7        # 1 val + 6 train, ~700M tokens, ~1.4GB

def find_shards():
    # 1. attached Kaggle Dataset (persists across sessions, read-only, free)
    hits = glob.glob("/kaggle/input/**/edufineweb_*.npy", recursive=True)
    if hits:
        return os.path.dirname(hits[0]), "attached dataset"
    # 2. already downloaded in this session
    if glob.glob("edu_fineweb10B/edufineweb_*.npy"):
        return "edu_fineweb10B", "this session"
    return None, None

DATA_DIR, source = find_shards()
if DATA_DIR is None:
    print(f"no shards found, downloading {SHARDS} (this is the slow path -- "
          f"save it as a Dataset afterwards so you only pay once)")
    subprocess.run([sys.executable, "fineweb.py", "--shards", str(SHARDS)], check=True)
    DATA_DIR, source = find_shards()

files = sorted(os.path.basename(f) for f in glob.glob(f"{DATA_DIR}/edufineweb_*.npy"))
n_train = sum("train" in f for f in files)
print(f"using {DATA_DIR}  (from: {source})")
print(f"{len(files)} shards: {n_train} train, {len(files)-n_train} val "
      f"= ~{n_train*100}M train tokens")
for f in files:
    print("  ", f)

## 6. Define the ablations

One variable changes per row. `learned + layernorm` is the GPT-2 control; `rope +
layernorm` isolates RoPE; `rope + rmsnorm` adds RMSNorm on top.

In [ ]:
MAX_STEPS = 1200          # x 131072 tokens = ~157M tokens per run
                          # Sized from measured throughput, not optimism: at the ~8.8K
                          # tok/s first observed on a T4, 4000 steps was 16.5h per run
                          # and 33h total -- past both the 12h session cap and the 30h
                          # weekly quota. Raise this only after the probe cell below
                          # tells you what your throughput actually is.

WARMUP = max(20, MAX_STEPS // 27)   # ~3.75% of training, same fraction nanoGPT uses

tokens_per_run = MAX_STEPS * 131072
print(f"each run consumes {tokens_per_run/1e6:.0f}M tokens; "
      f"{n_train*100}M available -> {tokens_per_run/(n_train*1e8):.2f} epochs")
print(f"warmup {WARMUP} steps ({WARMUP/MAX_STEPS*100:.1f}% of training), "
      f"peak lr 6e-4, cosine to 6e-5")

COMMON = [
    f"--data_dir={DATA_DIR}",
    "--n_layer=6", "--n_head=6", "--n_embd=384", "--block_size=512",
    "--batch_size=16", "--seq_len=512", "--total_batch_size=131072",
    f"--max_steps={MAX_STEPS}", f"--warmup_steps={WARMUP}",
    # eval often enough that a 1200-step run still gives a usable validation curve
    "--eval_every=100", "--val_steps=20", "--ckpt_every=100",
    "--resume",           # no-op on a fresh run, restart-safe on a resumed one
]

RUNS = [
    {"name": "baseline_learned_layernorm", "pos": "learned", "norm": "layernorm"},
    {"name": "rope_layernorm",             "pos": "rope",    "norm": "layernorm"},
    {"name": "modern_rope_rmsnorm",        "pos": "rope",    "norm": "rmsnorm"},
]

for r in RUNS:
    print(f"{r['name']:<30} pos={r['pos']:<8} norm={r['norm']}")

## 7a. Throughput probe — decides sequential vs parallel

Running two models at once only helps if a single process is not already saturating the
machine. It is not obvious which regime you are in: a first T4 attempt here measured
**8.8K tok/s per process with two running**, about 2.7x slower than the same model on a
laptop RTX 3050, which points at the two processes contending for Kaggle's 4 vCPUs rather
than at the GPU.

This cell measures one process alone, then projects both strategies. Sixty seconds now
decides where ten hours go.

In [ ]:
from train_modern_gpt import GPT, GPTConfig, pick_amp_dtype

def probe(dtype, B=16, T=512, iters=10):
    m = GPT(GPTConfig(block_size=512, vocab_size=50304, n_layer=6, n_head=6,
                      n_embd=384, pos="rope", norm="rmsnorm")).cuda()
    opt = m.configure_optimizers(0.1, 6e-4, "cuda", verbose=False)
    scaler = torch.amp.GradScaler("cuda", enabled=(dtype == torch.float16))
    x = torch.randint(0, 50304, (B, T), device="cuda")
    y = torch.randint(0, 50304, (B, T), device="cuda")

    def step():
        opt.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=dtype):
            _, loss = m(x, y)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
        scaler.step(opt); scaler.update()

    for _ in range(3):
        step()
    torch.cuda.synchronize(); t0 = time.perf_counter()
    for _ in range(iters):
        step()
    torch.cuda.synchronize()
    dt = (time.perf_counter() - t0) / iters
    del m, opt, x, y
    torch.cuda.empty_cache()
    return B * T / dt, dt

# Measure both rather than trusting any capability API. On a T4 fp16 should win by a
# wide margin; if bf16 looks competitive you are on Ampere or later.
results = {}
for name, dt_ in (("float16", torch.float16), ("bfloat16", torch.bfloat16)):
    try:
        tok_s, ms = probe(dt_)
        results[name] = tok_s
        print(f"{name:>9}: {tok_s:>9,.0f} tok/s   ({ms*1000:6.0f} ms / 8192-token micro-step)")
    except RuntimeError as e:
        print(f"{name:>9}: failed ({str(e)[:60]})")

chosen = pick_amp_dtype("cuda")
print(f"\ntrainer picks {chosen} automatically")
if len(results) == 2:
    best = max(results, key=results.get)
    print(f"fastest measured: {best} ({results[best]/min(results.values()):.1f}x the other)")
    if str(chosen).split(".")[-1] != best:
        print(f"MISMATCH -- override with --dtype={best}")

SOLO_TOK_S = max(results.values())

OBSERVED_PARALLEL_TOK_S = 8809   # per-process with 2 running; replace with your own
n_gpus = max(1, torch.cuda.device_count())
seq_h = len(RUNS) * MAX_STEPS * (131072 / SOLO_TOK_S) / 3600
par_waves = math.ceil(len(RUNS) / n_gpus)
par_h = par_waves * MAX_STEPS * (131072 / OBSERVED_PARALLEL_TOK_S) / 3600
print()
print(f"SEQUENTIAL: {len(RUNS)} runs x {MAX_STEPS} steps = {seq_h:.1f} h")
print(f"PARALLEL  : {par_waves} waves x {MAX_STEPS} steps = {par_h:.1f} h "
      f"(using the older 2-at-once number)")
print(f"-> use {'SEQUENTIAL' if seq_h < par_h else 'PARALLEL'}")
if min(seq_h, par_h) > 11.5:
    print(f"WARNING: {min(seq_h, par_h):.1f} h exceeds the ~12h session cap -- "
          f"lower MAX_STEPS or plan a second commit (--resume handles it).")

## 7b. Train

A tiny scheduler pins each run to one GPU via `CUDA_VISIBLE_DEVICES` and hands the next
queued run to whichever GPU frees up. `SEQUENTIAL = True` restricts it to one GPU, so the
runs go strictly one after another — set it from what the probe just told you.

Progress is polled rather than streamed, because two processes writing to one notebook
stdout interleave into noise. Full output is in `log/<run>/stdout.txt`.

In [ ]:
SEQUENTIAL = True         # one model at a time; set False to use both T4s at once

GPUS = [0] if SEQUENTIAL else list(range(max(1, torch.cuda.device_count())))
POLL_SECONDS = 120

def launch(run, gpu):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    cmd = [sys.executable, "-u", "train_modern_gpt.py",
           f"--pos={run['pos']}", f"--norm={run['norm']}",
           f"--run_name={run['name']}"] + COMMON
    os.makedirs(f"log/{run['name']}", exist_ok=True)
    fh = open(f"log/{run['name']}/stdout.txt", "a")
    fh.write(f"\n===== launched on GPU {gpu} at {time.strftime('%H:%M:%S')} =====\n")
    fh.flush()
    proc = subprocess.Popen(cmd, stdout=fh, stderr=subprocess.STDOUT, env=env)
    print(f"[GPU {gpu}] started {run['name']} (pid {proc.pid})")
    return proc, fh

def last_step_line(name):
    path = f"log/{name}/stdout.txt"
    if not os.path.exists(path):
        return "(no output yet)"
    with open(path, errors="ignore") as f:
        lines = [l for l in f if l.startswith("step ") or l.startswith("validation")]
    return lines[-1].strip() if lines else "(starting up)"

def train_all(runs, gpus=GPUS):
    pending, active = list(runs), {}
    t_start = time.time()
    while pending or active:
        for gpu in gpus:
            if gpu not in active and pending:
                run = pending.pop(0)
                active[gpu] = (run, *launch(run, gpu))
        time.sleep(POLL_SECONDS)
        elapsed = (time.time() - t_start) / 60
        print(f"\n--- {elapsed:.0f} min elapsed ---")
        for gpu, (run, proc, fh) in sorted(active.items()):
            print(f"[GPU {gpu}] {run['name']}: {last_step_line(run['name'])}")
        for gpu, (run, proc, fh) in list(active.items()):
            if proc.poll() is not None:
                fh.close()
                status = "finished" if proc.returncode == 0 else f"FAILED rc={proc.returncode}"
                print(f"[GPU {gpu}] {run['name']} {status}")
                del active[gpu]
    print(f"\nall runs done in {(time.time()-t_start)/60:.0f} min")

train_all(RUNS)

## 8. Results

Parses `log/<run>/log.txt` into loss curves and the comparison table. If a session timed
out mid-run, re-run the training cell first — everything resumes from its checkpoint.

In [ ]:
import matplotlib.pyplot as plt

def read_log(name):
    train, val = [], []
    path = f"log/{name}/log.txt"
    if not os.path.exists(path):
        return train, val
    for line in open(path):
        parts = line.split()
        if len(parts) == 3 and parts[1] in ("train", "val"):
            (train if parts[1] == "train" else val).append((int(parts[0]), float(parts[2])))
    return train, val

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for run in RUNS:
    train, val = read_log(run["name"])
    if train:
        axes[0].plot(*zip(*train), label=run["name"], alpha=0.8, lw=1)
    if val:
        axes[1].plot(*zip(*val), label=run["name"], marker="o", ms=3)
axes[0].set(title="Training loss", xlabel="step", ylabel="loss")
axes[1].set(title="Validation loss", xlabel="step", ylabel="loss")
for ax in axes:
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("loss_curves.png", dpi=140)
plt.show()

In [ ]:
# Comparison table, ready to paste into the README
print("| Model | RoPE | Norm | Params | Val loss | PPL |")
print("| :--- | :---: | :--- | ---: | ---: | ---: |")
for run in RUNS:
    _, val = read_log(run["name"])
    ckpt = f"log/{run['name']}/ckpt.pt"
    n_params = ""
    if os.path.exists(ckpt):
        sd = torch.load(ckpt, map_location="cpu", weights_only=False)["model"]
        # dedupe by storage: wte.weight and lm_head.weight are tied, so they appear
        # as two state_dict entries backed by one tensor. Summing naively adds the
        # 50304 x 384 embedding twice and over-reports by 19.3M parameters.
        seen, n = set(), 0
        for v in sd.values():
            if v.data_ptr() not in seen:
                seen.add(v.data_ptr()); n += v.numel()
        n_params = f"{n:,}"
    if val:
        best = min(v for _, v in val)
        print(f"| {run['name']} | {'yes' if run['pos']=='rope' else 'no'} | {run['norm']} "
              f"| {n_params} | {best:.4f} | {math.exp(best):.2f} |")
    else:
        print(f"| {run['name']} | {'yes' if run['pos']=='rope' else 'no'} | {run['norm']} "
              f"| {n_params} | (no eval yet) | |")

## 9. Inference benchmarks

KV cache on/off against the trained modern model. Speed and memory do not depend on the
weights, but running against a real checkpoint keeps the numbers honest.

In [ ]:
!python benchmark.py --mode=inference --ckpt=log/modern_rope_rmsnorm/ckpt.pt \
    --batch_size=8 --shapes="64:128,128:256,256:256"

In [ ]:
# training throughput per architecture, one row each
for run in RUNS:
    print(f"### {run['name']}")
    subprocess.run([sys.executable, "benchmark.py", "--mode=training",
                    f"--pos={run['pos']}", f"--norm={run['norm']}",
                    "--n_layer=6", "--n_head=6", "--n_embd=384", "--block_size=512",
                    "--batch_size=8", "--seq_len=512", "--repeats=5"])

## 10. Keep the results

`/kaggle/working` is capped at 20GB and is what gets saved when the notebook commits.
Checkpoints are the bulk of it — drop them here if you only need the curves and logs.

In [ ]:
total = sum(os.path.getsize(p) for p in glob.glob("log/**/*", recursive=True) if os.path.isfile(p))
print(f"log/ is {total/1e6:.0f} MB")
for p in sorted(glob.glob("log/*/ckpt.pt")):
    print(f"  {p}  {os.path.getsize(p)/1e6:.0f} MB")

# Uncomment to drop checkpoints and keep only logs + curves:
# for p in glob.glob("log/*/ckpt.pt"):
#     os.remove(p)